# arange-fancy-index-cross-entropy composite — cx19: pick per-sample target logits via logits[arange(B), target]

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `arange-fancy-index-cross-entropy`, `index-by-tensor`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "arange-fancy-index-cross-entropy"
DD_ATOM_IDS = ["arange-fancy-index-cross-entropy", "index-by-tensor"]
DD_SUBTOPICS = ["Loss: arange fancy-index cross-entropy", "PyTorch: index by tensor"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`index-by-tensor` is the general atom: you index a tensor with another tensor and PyTorch returns the gathered values shaped by the index. `arange(B)` paired with a `target` of shape `(B,)` is the specific cross-entropy use case — `logits[arange(B), target]` returns a 1-D `(B,)` slice picking ONE column per row.

The composition exercises both atoms together: arange-fancy-index is the IDIOM, index-by-tensor is the MECHANISM. Get the mechanism wrong (e.g. `logits[:, target]`) and you get a (B, B) broadcasting result instead of the per-sample (B,) vector you wanted.

### Composite Exercise — pick per-sample target logits via logits[arange(B), target]

**Atoms exercised together**: `arange-fancy-index-cross-entropy`, `index-by-tensor`

Implement `cx19_pick_target_logits(logits, target)` that:

- Takes `logits` of shape `(B, C)` and `target` of shape `(B,)` (dtype long).
- Returns the per-sample target logit of shape `(B,)` via `logits[arange(B), target]`.

The output must NOT be `(B, B)` — that would mean you wrote `logits[:, target]` (broadcasting) instead of the arange-paired form.

In [ ]:
def cx19_pick_target_logits(logits, target):
    # arange-fancy-index idiom: index dim-0 by arange(B), dim-1 by target.
    # Both are 1-D tensors of length B → output is 1-D length B (NOT broadcast).
    B = logits.shape[0]
    return logits[t.arange(B), target]


<details><summary>Show solution — cx19</summary>

```python
def cx19_pick_target_logits(logits, target):
    # arange-fancy-index idiom: index dim-0 by arange(B), dim-1 by target.
    # Both are 1-D tensors of length B → output is 1-D length B (NOT broadcast).
    B = logits.shape[0]
    return logits[t.arange(B), target]
```

The arange-pairing is what suppresses broadcasting. `logits[:, target]` would broadcast the (B,)-shaped target across the (B,) row axis, producing (B, B). Pairing target with `arange(B)` (also length B) tells PyTorch you want a 1-D gather, not a 2-D cross-product.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx19'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx19',
        'subtopics': ["Loss: arange fancy-index cross-entropy", "PyTorch: index by tensor"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()